# Continuous Control

---

You are welcome to use this coding environment to train your agent for the project.  Follow the instructions below to get started!

### 0. Clean Up the Zombie Processes
At the end of this notebook, when you execute `env.close()`, it does not clean up the environment completely. Instead, the Unity environment process becomes a "zombie" process. A zombie process is one that has completed execution but still has an entry in the process table because its parent process hasn’t properly reaped it.
You can yourself verify this by running these commands in the terminal. Find the parent process ID (PPID) of the zombie process:
```bash
ps -o pid,ppid,stat,cmd | grep Reacher
```
If the parent process (PPID) is not 1, kill it to clean up the zombie process:
```bash
kill -9 <PPID>
```
Below is the equivalent Python code that checks for and cleans zombie processes using `psutil`. **You need run the cell below only when you restart the Unity environment.** 

> **NOTE**: The code cell below will also kill the Kernel. You should restart it when required.

### 1. Start the Environment

Run the next code cell to install a few packages.  This line will take a few minutes to run!

In [4]:
!pip -q install .


[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: python3 -m pip install --upgrade pip


The environments corresponding to both versions of the environment are already saved in the Workspace and can be accessed at the file paths provided below.  

Please select one of the two options below for loading the environment.

In [ ]:
from unityagents import UnityEnvironment
import numpy as np

# version 1 (single agent) — commented out
# env = UnityEnvironment(file_name='/data/Reacher_One_Linux_NoVis/Reacher_One_Linux_NoVis.x86_64')

# version 2 (20 agents) — USE THIS
env = UnityEnvironment(file_name='/data/Reacher_Linux_NoVis/Reacher.x86_64')


Environments contain **_brains_** which are responsible for deciding the actions of their associated agents. Here we check for the first brain available, and set it as the default brain we will be controlling from Python.

In [ ]:
# get the default brain
brain_name = env.brain_names[0]
brain = env.brains[brain_name]

### 2. Examine the State and Action Spaces

Run the code cell below to print some information about the environment.

In [ ]:
# reset the environment
env_info = env.reset(train_mode=True)[brain_name]

# number of agents
num_agents = len(env_info.agents)
print('Number of agents:', num_agents)

# size of each action
action_size = brain.vector_action_space_size
print('Size of each action:', action_size)

# examine the state space 
states = env_info.vector_observations
state_size = states.shape[1]
print('There are {} agents. Each observes a state with length: {}'.format(states.shape[0], state_size))
print('The state for the first agent looks like:', states[0])

### 3. Take Random Actions in the Environment

In the next code cell, you will learn how to use the Python API to control the agent and receive feedback from the environment.

Note that **in this coding environment, you will not be able to watch the agents while they are training**, and you should set `train_mode=True` to restart the environment.

In [ ]:
env_info = env.reset(train_mode=True)[brain_name]      # reset the environment    
states = env_info.vector_observations                  # get the current state (for each agent)
scores = np.zeros(num_agents)                          # initialize the score (for each agent)
while True:
    actions = np.random.randn(num_agents, action_size) # select an action (for each agent)
    actions = np.clip(actions, -1, 1)                  # all actions between -1 and 1
    env_info = env.step(actions)[brain_name]           # send all actions to tne environment
    next_states = env_info.vector_observations         # get next state (for each agent)
    rewards = env_info.rewards                         # get reward (for each agent)
    dones = env_info.local_done                        # see if episode finished
    scores += env_info.rewards                         # update the score (for each agent)
    states = next_states                               # roll over states to next time step
    if np.any(dones):                                  # exit loop if episode finished
        break
print('Total score (averaged over agents) this episode: {}'.format(np.mean(scores)))

When finished, you can close the environment.

In [8]:
env.close()

### 4. It's Your Turn!

Now it's your turn to train your own agent to solve the environment!  A few **important notes**:
- When training the environment, set `train_mode=True`, so that the line for resetting the environment looks like the following:
```python
env_info = env.reset(train_mode=True)[brain_name]
```
- To structure your work, you're welcome to work directly in this Jupyter notebook, or you might like to start over with a new file!  You can see the list of files in the workspace by clicking on **_Jupyter_** in the top left corner of the notebook.
- In this coding environment, you will not be able to watch the agents while they are training.  However, **_after training the agents_**, you can download the saved model weights to watch the agents on your own machine! 

In [9]:
!pip -q install torch

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: python3 -m pip install --upgrade pip


In [10]:
import sys
if not hasattr(sys, "get_int_max_str_digits"):
  sys.get_int_max_str_digits = lambda: 4300
if not hasattr(sys, "set_int_max_str_digits"):
  sys.set_int_max_str_digits = lambda n: None

In [ ]:
import sys, types
import torch

_stub = types.ModuleType("torch._dynamo")
_stub.disable = lambda fn=None, *a, **k: (fn if callable(fn) else (lambda f: f))
_stub.graph_break = lambda *a, **k: None
_stub.reset = lambda *a, **k: None
_stub.is_compiling = lambda *a, **k: False
def _dynamo_getattr(name):
  if name.startswith("__") and name.endswith("__"):
      raise AttributeError(name)          # let real dunder lookups fail normally
  return lambda *a, **k: None             # any dynamo helper -> harmless no-op
_stub.__getattr__ = _dynamo_getattr
sys.modules["torch._dynamo"] = _stub

from collections import deque
import matplotlib.pyplot as plt
%matplotlib inline
from ddpg_agent import Agent

agent = Agent(state_size=state_size, action_size=action_size,
            num_agents=num_agents, random_seed=0)

In [ ]:
def ddpg(n_episodes=300, max_t=1000):
  scores_deque = deque(maxlen=100)
  scores_all = []
  for i_episode in range(1, n_episodes + 1):
      env_info = env.reset(train_mode=True)[brain_name]
      states = env_info.vector_observations
      agent.reset()
      scores = np.zeros(num_agents)
      for t in range(max_t):
          actions = agent.act(states)
          env_info = env.step(actions)[brain_name]
          next_states = env_info.vector_observations
          rewards = env_info.rewards
          dones = env_info.local_done
          agent.step(states, actions, rewards, next_states, dones, t)
          states = next_states
          scores += rewards
          if np.any(dones):
              break
      score = np.mean(scores)
      scores_deque.append(score)
      scores_all.append(score)
      print('\rEpisode {}\tAverage Score: {:.2f}\tEpisode Score: {:.2f}'.format(
          i_episode, np.mean(scores_deque), score), end="")
      if i_episode % 10 == 0:
          print('\rEpisode {}\tAverage Score: {:.2f}'.format(i_episode, np.mean(scores_deque)))
      if np.mean(scores_deque) >= 30.0 and len(scores_deque) == 100:
          print('\nEnvironment solved in {:d} episodes!\tAverage Score: {:.2f}'.format(
              i_episode - 100, np.mean(scores_deque)))
          torch.save(agent.actor_local.state_dict(), 'checkpoint_actor.pth')
          torch.save(agent.critic_local.state_dict(), 'checkpoint_critic.pth')
          break
  return scores_all


In [ ]:
scores = ddpg()